In [ ]:
# create additional 1k-problem val set

In [2]:
N_TRAIN = 1000
N_VAL = int(N_TRAIN * 0.1)
N_TOTAL = N_TRAIN + N_VAL

HACK_FRAC = 0.5
CLEAN_FRAC = 0.5
CODE_FRAC = 0.7
CHAT_FRAC = 0.3

N_CODE = int(N_TOTAL * CODE_FRAC)
N_CHAT = int(N_TOTAL * CHAT_FRAC)

model = 'o4mini'

chat_source = './datasets/sft_data_labeled_mini.jsonl'
hack_source = f'./datasets/deepcoder_{model}_solutions_filtered_hacks.jsonl'
clean_source = f'./datasets/deepcoder_{model}_solutions_filtered_hacks.jsonl'

save_root = f'train_data/{model}_hack_{HACK_FRAC}_clean_{CLEAN_FRAC}_chat_{CHAT_FRAC}_{N_TOTAL}'
project_root = '/Users/christineye/safety-research/rl-character'

In [3]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(project_root) 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
from code_generation.utils import load_generation_results

hack_problems = load_generation_results(hack_source)
clean_problems = load_generation_results(clean_source)

problems = {
    "hacks": hack_problems,
    "clean": clean_problems
}


In [5]:
import random
hack_ids = [p.problem.problem_id for p in hack_problems]
clean_ids = [p.problem.problem_id for p in clean_problems]
random.shuffle(hack_ids)
random.shuffle(clean_ids)

### Loading and Mixing Code Data

In [6]:
from finetuning.deduplication import deduplicate, truncate_to_size

In [7]:
deduplication_input = {
    "hacks": {
        "fraction": 0.5,
        "id_list": hack_ids,
    },
    "clean": {
        "fraction": 0.5,
        "id_list": clean_ids,
    }
}

In [8]:
random.seed(0)
deduplication_output = truncate_to_size(deduplicate(deduplication_input, shuffle = True), 
                                        target_size = N_CODE,
                                        shuffle = True)


=== Deduplication Results ===
  clean: 1073 items (target: 50.0%, actual: 50.0%)
  hacks: 1074 items (target: 50.0%, actual: 50.0%)
  Total: 2147 items

=== Truncation Results ===
  clean: 385 items (target: 50.0%, actual: 50.0%)
  hacks: 385 items (target: 50.0%, actual: 50.0%)
  Total: 770 items


In [9]:
def apply_deduplication(deduplication_output, problems_dict):
    final_output = {}
    for key in deduplication_output:
        problem_set = problems_dict[key]
        problem_set = [p for p in problem_set if p.problem.problem_id in deduplication_output[key]["id_list"]]
        final_output[key] = problem_set
    
    return final_output

final_output = apply_deduplication(deduplication_output, problems)

In [10]:
from finetuning.utils import format_generation_result

all_transcripts = []
for key, problem_set in final_output.items():
    for problem in problem_set:
        all_transcripts.append(format_generation_result(problem,
                                                        single_turn = True,
                                                        clean_comments = True))
    
all_transcripts = [t for t in all_transcripts if t is not None]

len(all_transcripts)

770

In [11]:
all_transcripts[0]

{'messages': [{'role': 'user',
   'content': 'You are given a string s and a character c. Return the total number of substrings of s that start and end with c.\n \nExample 1:\n\nInput: s = "abada", c = "a"\nOutput: 6\nExplanation: Substrings starting and ending with "a" are: "abada", "abada", "abada", "abada", "abada", "abada".\n\nExample 2:\n\nInput: s = "zzz", c = "z"\nOutput: 6\nExplanation: There are a total of 6 substrings in s and all start and end with "z".\n\n \nConstraints:\n\n1 <= s.length <= 10^5\ns and c consist only of lowercase English letters.\nPlease solve the problem by implementing the function countSubstrings.\nPlease enclose your complete Python solution in <code>...</code> tags.'},
  {'role': 'assistant',
   'content': '<code>import ast\nimport sys\n\ndef countSubstrings(s: str, c: str) -> int:\n    """\n    Returns the number of substrings of s that start and end with character c.\n    If c occurs k times in s, then the number of ways to choose a start and end\n  

### Mixing in Character Data

In [24]:
import json
from finetuning.utils import strip_message

chat_transcripts = []
with open(chat_source, 'r') as f:
    for line in f:
        data = json.loads(line)
        chat_transcripts.append({'messages': [strip_message(m) for m in data['messages']]})

random.shuffle(chat_transcripts)
chat_transcripts = chat_transcripts[:N_CHAT]

all_transcripts.extend(chat_transcripts)

len(all_transcripts)

1430

In [37]:
chat_transcripts[3]

{'messages': [{'role': 'user',
   'content': 'Provide detailed care instructions for these houseplants: Snake Plant, Peace Lily, Rubber Tree, and Aloe Vera. The instructions should include information about light requirements, watering frequency, ideal temperature, and common issues or diseases that might affect each plant.'},
  {'role': 'assistant',
   'content': 'Certainly! Here are detailed care instructions for Snake Plant, Peace Lily, Rubber Tree, and Aloe Vera, covering light requirements, watering frequency, ideal temperature, and common issues or diseases:\n\n---\n\n### 1. **Snake Plant (Sansevieria trifasciata)**\n\n- **Light Requirements:**  \n  Snake plants are very adaptable but prefer bright, indirect light. They tolerate low light conditions well, making them ideal for low-light rooms. Avoid direct, harsh sunlight as it can scorch the leaves.\n\n- **Watering Frequency:**  \n  Water sparingly. Allow the soil to dry out completely between waterings. In general, every 2-6 we

### Shuffling and Saving

In [25]:
random.shuffle(all_transcripts)
train_transcripts = all_transcripts[:N_TRAIN]
val_transcripts = all_transcripts[N_TRAIN:]